# Position-aware FinCast Controller

This notebook keeps FinCast frozen and trains a separate recurrent position controller. The controller consumes one FinCast predictive distribution patch `[H, 10]`, remembers the previous position through a GRU state, and outputs the next long-only position percentage `p_t in [0, 1]`.

The default path uses synthetic distribution patches as a smoke test. Set `RUN_FINCAST = True` to build a real cached feature file from the frozen FinCast checkpoint.

Run this notebook with the FinCast environment, for example `.conda-fincast`, or any kernel that has `torch` installed.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

WORKSPACE_ROOT = Path.cwd()
if WORKSPACE_ROOT.name == "Inference":
    WORKSPACE_ROOT = WORKSPACE_ROOT.parent.parent
elif WORKSPACE_ROOT.name == "FinCast-fts":
    WORKSPACE_ROOT = WORKSPACE_ROOT.parent

FINCAST_ROOT = WORKSPACE_ROOT / "FinCast-fts"
FINCAST_SRC = FINCAST_ROOT / "src"

for path in (WORKSPACE_ROOT, FINCAST_SRC):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from src.utils.config import PositionControllerConfig, PositionLossConfig
from src.trader.cnn_gru import PositionAwareGRUPolicy
from src.training.losses import MeanVarianceTurnoverLoss
from src.datasets.trader_dataset import CachedDistributionDataset
from src.datasets.sources import DEFAULT_ETF_TICKERS, download_yfinance_close, save_close_csv
from src.fincast_io.forecast_features import forecast_to_return_patch, quantile_summary_features
from src.fincast_io.cache_builder import build_fincast_distribution_cache, load_distribution_cache
from src.baselines.baselines_legacy import closed_form_markowitz_position
from src.training.trainer import train_one_epoch, evaluate_policy

torch.set_float32_matmul_precision("high") if hasattr(torch, "set_float32_matmul_precision") else None
print(f"workspace: {WORKSPACE_ROOT}")
print(f"fincast:   {FINCAST_ROOT}")

## Config

Keep these knobs explicit. The first pass should stay small and controllable.

In [ ]:
RUN_DOWNLOAD_DATA = False
RUN_FINCAST = False
SEED = 7

TICKERS = list(DEFAULT_ETF_TICKERS)
DATA_START = "2007-01-01"
DATA_END = None
DATA_FREQUENCY = "D"
CSV_PATH = WORKSPACE_ROOT / "data" / "raw" / "etf_daily_close.csv"
FALLBACK_CSV_PATH = FINCAST_ROOT / "Inference" / "sample_close_monthly.csv"
MODEL_PATH = WORKSPACE_ROOT / "models" / "FinCast" / "v1.pth"
CACHE_PATH = WORKSPACE_ROOT / "data" / "cache" / "position_fincast_cache.csv"

CONTEXT_LEN = 128
HORIZON_LEN = 32
HOLDING_HORIZON = 1
SEQ_LEN = 32
BATCH_SIZE = 4
EPOCHS = 5
FINCAST_STRIDE = 5
MAX_WINDOWS_PER_ASSET = 256
FINCAST_BATCH_SIZE = 16

LEARNING_RATE = 1e-3
LAMBDA_VARIANCE = 1.0
LAMBDA_TURNOVER = 0.01
LAMBDA_FORECAST_RISK = 0.1
RISK_AVERSION_BASELINE = 10.0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
print(f"device: {DEVICE}")
print(f"target csv: {CSV_PATH}")

## Data Source: ETF Daily Close

First-version data is a wide daily close CSV from yfinance: `Date, SPY, QQQ, ...`. Keep `RUN_DOWNLOAD_DATA=False` unless you want this notebook to hit the network; the same download is also available as `python scripts/download_prices.py`.

In [ ]:
if RUN_DOWNLOAD_DATA:
    df_close = download_yfinance_close(TICKERS, start=DATA_START, end=DATA_END, auto_adjust=True)
    save_close_csv(df_close, CSV_PATH)
    print(f"downloaded {len(df_close)} rows -> {CSV_PATH}")
elif CSV_PATH.exists():
    df_close = pd.read_csv(CSV_PATH)
    print(f"using existing data: {CSV_PATH} ({len(df_close)} rows)")
else:
    print(f"missing real ETF data: {CSV_PATH}")
    print(r"download with: .\.conda-fincast\python.exe scripts\download_prices.py")
    df_close = pd.read_csv(FALLBACK_CSV_PATH)
    print(f"notebook smoke fallback available: {FALLBACK_CSV_PATH} ({len(df_close)} rows)")

display(df_close.head())

## Optional: Cache Frozen FinCast Features

This cell does not train or modify FinCast. It runs the frozen checkpoint on controlled rolling windows and saves `[H, 10]` distribution patches. `FINCAST_STRIDE` and `MAX_WINDOWS_PER_ASSET` keep the first version manageable.

In [ ]:
if RUN_FINCAST:
    if not CSV_PATH.exists():
        raise FileNotFoundError(f"Missing ETF data CSV: {CSV_PATH}. Run scripts/download_prices.py first.")
    if not MODEL_PATH.exists():
        raise FileNotFoundError(f"Missing FinCast checkpoint: {MODEL_PATH}")

    build_fincast_distribution_cache(
        csv_path=CSV_PATH,
        model_path=MODEL_PATH,
        output_path=CACHE_PATH,
        fincast_root=FINCAST_ROOT,
        tickers=TICKERS,
        context_len=CONTEXT_LEN,
        horizon_len=HORIZON_LEN,
        holding_horizon=HOLDING_HORIZON,
        data_frequency=DATA_FREQUENCY,
        stride=FINCAST_STRIDE,
        max_windows_per_asset=MAX_WINDOWS_PER_ASSET,
        batch_size=FINCAST_BATCH_SIZE,
        backend="gpu" if torch.cuda.is_available() else "cpu",
    )
    print(f"saved cache: {CACHE_PATH}")
else:
    print("Skipping FinCast feature cache. Using cached file if present; otherwise synthetic smoke data.")

## Load Cached Patches or Build Synthetic Smoke Data

`return_patches` has shape `[T, H, 10]`: channel 0 is mean, channels 1-9 are quantiles. The controller sees this full distribution patch, not a hand-compressed one-point feature vector.

In [ ]:
def build_synthetic_distribution_patches(T=192, H=32, seed=7):
    rng = np.random.default_rng(seed)
    shocks = 0.002 + 0.025 * rng.standard_normal(T + H + 1)
    values = 100.0 * np.cumprod(1.0 + shocks)
    last_values = values[:T].astype(np.float32)
    realized_returns = (values[HOLDING_HORIZON:T + HOLDING_HORIZON] / values[:T] - 1.0).astype(np.float32)

    horizon_scale = np.linspace(0.25, 1.0, H, dtype=np.float32)[None, :, None]
    signal = realized_returns[:, None, None] * horizon_scale
    noise = 0.004 * rng.standard_normal((T, H, 1)).astype(np.float32)
    mean_path = signal + noise

    spread = (0.015 + 0.04 * np.abs(rng.standard_normal((T, 1, 1)))).astype(np.float32)
    z_scores = np.asarray([-1.28, -0.84, -0.52, -0.25, 0.0, 0.25, 0.52, 0.84, 1.28], dtype=np.float32)
    quantiles = mean_path + spread * z_scores.reshape(1, 1, -1)
    return_patch = np.concatenate([mean_path, quantiles], axis=-1).astype(np.float32)
    full_outputs = last_values[:, None, None] * (1.0 + return_patch)
    return full_outputs.astype(np.float32), last_values, realized_returns

if CACHE_PATH.exists():
    cache = load_distribution_cache(CACHE_PATH)
    full_outputs = cache["full_outputs"].astype(np.float32)
    last_values = cache["last_values"].astype(np.float32)
    realized_returns = cache["realized_returns"].astype(np.float32)
    source_name = f"FinCast cache: {CACHE_PATH}"
else:
    full_outputs, last_values, realized_returns = build_synthetic_distribution_patches(
        T=192,
        H=HORIZON_LEN,
        seed=SEED,
    )
    source_name = "synthetic smoke data"

return_patches = forecast_to_return_patch(full_outputs, last_values).float()
realized_returns = torch.as_tensor(realized_returns, dtype=torch.float32)
summary = quantile_summary_features(return_patches)
forecast_risk = summary[:, 4].clamp_min(0.0).pow(2)

print(source_name)
print("return_patches:", tuple(return_patches.shape))
print("realized_returns:", tuple(realized_returns.shape))
print("forecast_risk:", tuple(forecast_risk.shape))

## Dataset

Each training sample is a contiguous sequence. The GRU state rolls forward inside the sequence, and the delta head controls how much the position can change at each step.

In [ ]:
T = return_patches.shape[0]
split = int(T * 0.7)

train_ds = CachedDistributionDataset(
    return_patches[:split],
    realized_returns[:split],
    seq_len=SEQ_LEN,
    forecast_risk=forecast_risk[:split],
    stride=max(1, SEQ_LEN // 2),
)
valid_ds = CachedDistributionDataset(
    return_patches[split - SEQ_LEN:],
    realized_returns[split - SEQ_LEN:],
    seq_len=SEQ_LEN,
    forecast_risk=forecast_risk[split - SEQ_LEN:],
    stride=SEQ_LEN,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"train sequences: {len(train_ds)}, valid sequences: {len(valid_ds)}")

## Model

The patch encoder reads the full `[H, 10]` distribution curve. The GRUCell is the position-aware core: it receives the encoded patch plus the previous position, updates its hidden state, and emits a bounded position change.

In [ ]:
controller_cfg = PositionControllerConfig(
    horizon_len=return_patches.shape[1],
    forecast_channels=return_patches.shape[2],
    encoder_dim=64,
    conv_hidden=64,
    conv_layers=2,
    kernel_size=3,
    state_dim=64,
    dropout=0.1,
    max_trade=0.25,
    min_position=0.0,
    max_position=1.0,
)
loss_cfg = PositionLossConfig(
    lambda_variance=LAMBDA_VARIANCE,
    lambda_turnover=LAMBDA_TURNOVER,
    lambda_forecast_risk=LAMBDA_FORECAST_RISK,
)

model = PositionAwareGRUPolicy(controller_cfg).to(DEVICE)
loss_fn = MeanVarianceTurnoverLoss(loss_cfg)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable params: {num_params:,}")

## Train

This optimizes realized mean-variance utility with turnover and optional forecast-risk exposure penalties. For real experiments, keep the chronological split and compare against the baselines below.

In [ ]:
history = []
for epoch in range(1, EPOCHS + 1):
    train_metrics = train_one_epoch(
        model,
        loss_fn,
        train_loader,
        optimizer,
        device=DEVICE,
        initial_position=0.0,
        grad_clip=1.0,
    )
    valid_metrics = evaluate_policy(
        model,
        loss_fn,
        valid_loader,
        device=DEVICE,
        initial_position=0.0,
    )
    row = {f"train_{k}": v for k, v in train_metrics.items()}
    row.update({f"valid_{k}": v for k, v in valid_metrics.items()})
    history.append(row)
    print(
        f"epoch {epoch:02d} | "
        f"train_loss={train_metrics['loss']:.6f} "
        f"valid_loss={valid_metrics['loss']:.6f} "
        f"valid_ret={valid_metrics['mean_return']:.6f} "
        f"valid_turnover={valid_metrics['turnover']:.4f}"
    )

pd.DataFrame(history)

## Baselines and Diagnostics

The learned controller should beat simple rules before adding complexity. The Markowitz baseline uses FinCast-derived expected return and quantile-spread risk.

In [ ]:
@torch.no_grad()
def rollout_all(model, patches):
    model.eval()
    rollout = model(patches.unsqueeze(0).to(DEVICE), initial_position=0.0)
    return rollout.positions.squeeze(0).cpu(), rollout.deltas.squeeze(0).cpu()

def metrics_for_positions(name, positions, returns):
    positions = torch.as_tensor(positions, dtype=torch.float32)
    returns = torch.as_tensor(returns, dtype=torch.float32)
    pnl = positions * returns
    turnover = torch.cat([positions[:1].abs(), (positions[1:] - positions[:-1]).abs()]).mean()
    vol = pnl.std(unbiased=False).clamp_min(1e-8)
    sharpe_like = pnl.mean() / vol
    return {
        "name": name,
        "mean_return": float(pnl.mean()),
        "volatility": float(vol),
        "sharpe_like": float(sharpe_like),
        "turnover": float(turnover),
        "avg_position": float(positions.mean()),
    }

positions_model, deltas_model = rollout_all(model, return_patches)
expected_return = summary[:, 3]  # q50 terminal return
markowitz_positions = closed_form_markowitz_position(
    expected_return,
    forecast_risk,
    risk_aversion=RISK_AVERSION_BASELINE,
)
buy_hold_positions = torch.ones_like(realized_returns)
flat_positions = torch.zeros_like(realized_returns)

report = pd.DataFrame([
    metrics_for_positions("learned_controller", positions_model, realized_returns),
    metrics_for_positions("markowitz_baseline", markowitz_positions, realized_returns),
    metrics_for_positions("buy_hold", buy_hold_positions, realized_returns),
    metrics_for_positions("flat", flat_positions, realized_returns),
])
report

In [ ]:
plot_n = min(120, len(realized_returns))
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axes[0].plot(realized_returns[:plot_n].numpy(), label="realized return", linewidth=1)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].legend(loc="best")

axes[1].plot(positions_model[:plot_n].numpy(), label="learned p_t", linewidth=2)
axes[1].plot(markowitz_positions[:plot_n].numpy(), label="markowitz p_t", linewidth=1.5, alpha=0.75)
axes[1].set_ylim(-0.05, 1.05)
axes[1].legend(loc="best")

axes[2].plot((positions_model[:plot_n] * realized_returns[:plot_n]).cumsum().numpy(), label="learned cumulative pnl")
axes[2].plot((markowitz_positions[:plot_n] * realized_returns[:plot_n]).cumsum().numpy(), label="markowitz cumulative pnl")
axes[2].legend(loc="best")
axes[2].set_xlabel("decision step")
plt.tight_layout()

## Next Controlled Extensions

- Replace synthetic smoke data with the FinCast cache by setting `RUN_FINCAST=True`.
- Keep the MLP/flatten baseline as an ablation if needed, but use this recurrent controller as the main position-aware unit.
- Tune `max_trade`, `lambda_turnover`, and `lambda_forecast_risk` before increasing model size.
- Only consider LoRA or modifying FinCast after this frozen-feature controller beats the simple Markowitz baseline on a chronological test split.